# Дообучение реранкера `BAAI/bge-reranker-v2-m3`

**Архитектура пайплайна RAG:**

```
Запрос пользователя
       │
       ▼
 Bi-encoder (retrieval)   ← USER-bge-m3 или e5-large
 top-50 кандидатов
       │
       ▼
 Cross-encoder (rerank)   ← bge-reranker-v2-m3  ← дообучаем здесь
 top-10 результатов
       │
       ▼
     LLM / ответ
```

**Принцип работы реранкера:**
- Bi-encoder кодирует запрос и документы **раздельно** → быстро (ANN-поиск)
- Cross-encoder принимает пару `[запрос, документ]` **вместе** → медленно, но точнее
- Реранкер переупорядочивает top-50 от bi-encoder → возвращает top-10

**Данные:** те же триплеты (anchor, positive, negative) — разворачиваем в пары с метками
```
(anchor, positive) → label = 1.0
(anchor, negative) → label = 0.0
```


## 0. Окружение

In [ ]:
!nvidia-smi 2>/dev/null || echo "CPU only"

In [ ]:
import sys, os, warnings, logging
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src import RerankerTrainer, RerankerConfig
from src.trainer import make_reranker_cpu_config, make_reranker_gpu_config
from src.data_preparation import Chunk, TripletSampler

DATA  = Path("../data")
OUT   = Path("../output/reranker");  OUT.mkdir(parents=True, exist_ok=True)

# Папка с bi-encoder результатами (для финального сравнения пайплайнов)
BIENCODER_OUT = Path("../output")

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# ═══════════════════════════════════════════════════════════════════════════════
# ПАРАМЕТРЫ
# ═══════════════════════════════════════════════════════════════════════════════
NUM_EPOCHS            = 3
EARLY_STOPPING        = 2
LOG_EVERY             = "epoch"

UNFREEZE_LAST_N_LAYERS = 6     # из 24 слоёв XLM-RoBERTa-large
DROPOUT_LAST_M_LAYERS  = 3
DROPOUT_P              = 0.1

# Сколько кандидатов отдавать реранкеру (retrieve_k) и сколько возвращать (rerank_k)
RETRIEVE_K = 50
RERANK_K   = 10
# ═══════════════════════════════════════════════════════════════════════════════

print(f"Устройство : {DEVICE}")
print(f"Модель     : {MODEL_NAME}")
if DEVICE == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")

## 1. Подготовка данных

Загружаем корпус и генерируем триплеты (те же, что для bi-encoder).
Реранкер обучается на **парах** — каждый триплет даёт 2 примера.


In [ ]:
corpus_df = pd.read_csv(DATA / "stage1_tsdae" / "corpus.csv")
print(f"Чанков: {len(corpus_df)} | Документов: {corpus_df['doc_id'].nunique()}")

chunks = [
    Chunk(text=row["sentence"], doc_id=row["doc_id"],
          chunk_id=row["chunk_id"], section=None)
    for row in corpus_df.to_dict("records")
]

# Генерируем триплеты: adjacent + query_template (реранкер учится
# сопоставлять реальные запросы с документами)
sampler = TripletSampler(chunks, negative_strategy="bm25", seed=42, max_chunks=3_000)
triplets_df = sampler.to_dataframe(
    strategies=["adjacent", "query_template"],
    n_per_chunk=1,
    max_chunks_qt=1_000,
)

print(f"\nТриплетов: {len(triplets_df)}")
print(triplets_df["strategy"].value_counts().to_string())

train_df, eval_df = train_test_split(triplets_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)
print(f"\nTrain: {len(train_df)} | Eval: {len(eval_df)}")

In [ ]:
# Что видит реранкер: пары (запрос/якорь, текст), метка 0 или 1
# Покажем несколько примеров
print("Примеры пар для реранкера:\n")
for _, row in train_df.sample(2, random_state=0).iterrows():
    print(f"Стратегия : {row['strategy']}")
    print(f"Anchor    : {row['anchor'][:120]}")
    print(f"Positive  : {row['positive'][:120]}  → label=1")
    print(f"Negative  : {row['negative'][:120]}  → label=0")
    print()

## 2. Конфигурация реранкера

`RerankerTrainer` внутри конвертирует триплеты в пары и обучает `CrossEncoder`
через `CrossEncoderTrainer` (ST >= 3.0) или через `CrossEncoder.fit()` (fallback).


In [ ]:
if DEVICE == "cpu":
    cfg_reranker = make_reranker_cpu_config(
        num_epochs=NUM_EPOCHS,
        output_dir=str(OUT),
    )
else:
    cfg_reranker = make_reranker_gpu_config(
        num_epochs=NUM_EPOCHS,
        output_dir=str(OUT),
    )

cfg_reranker.model_name              = MODEL_NAME
cfg_reranker.learning_rate           = 2e-5
cfg_reranker.warmup_ratio            = 0.1
cfg_reranker.early_stopping_patience = EARLY_STOPPING
cfg_reranker.logging_strategy        = LOG_EVERY
cfg_reranker.unfreeze_last_n_layers  = UNFREEZE_LAST_N_LAYERS
cfg_reranker.dropout_last_m_layers   = DROPOUT_LAST_M_LAYERS
cfg_reranker.dropout_p               = DROPOUT_P

print("Конфигурация реранкера:")
for k in ["model_name", "learning_rate", "num_epochs", "batch_size",
          "warmup_ratio", "unfreeze_last_n_layers", "dropout_last_m_layers", "dropout_p"]:
    print(f"  {k:30s}: {getattr(cfg_reranker, k)}")

## 3. Дообучение

In [ ]:
reranker_trainer = RerankerTrainer(cfg_reranker)
reranker_trainer.train(train_df, eval_df)

## 4. Оценка пайплайна: bi-encoder + reranker

Сравниваем три сценария на одних и тех же eval-запросах:
1. **Baseline** — базовый bi-encoder без дообучения, top-RERANK_K
2. **Bi-encoder only** — дообученный bi-encoder (TSDAE→MNRL), top-RERANK_K
3. **Bi-encoder + Reranker** — дообученный bi-encoder top-RETRIEVE_K → реранкер top-RERANK_K

Метрика: Recall@k (найден ли нужный документ в топ-k).


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def build_eval_corpus(df):
    """Строит корпус и ground-truth из триплетов."""
    corpus = list(set(df["positive"].tolist() + df["negative"].tolist()))
    corpus_idx = {t: i for i, t in enumerate(corpus)}
    return corpus, corpus_idx

def recall_at_k(scores_matrix, gt_indices, ks=(1, 3, 5, 10)):
    """scores_matrix: (n_queries, n_corpus), gt_indices: список правильных индексов."""
    results = {}
    n = len(gt_indices)
    for k in ks:
        hits = 0
        for i, gt in enumerate(gt_indices):
            top_k = np.argpartition(scores_matrix[i], -k)[-k:]
            if gt in top_k:
                hits += 1
        results[f"Recall@{k}"] = round(hits / n, 4)
    return results

corpus, corpus_idx = build_eval_corpus(eval_df)
queries  = eval_df["anchor"].tolist()
gt_idxs  = [corpus_idx.get(t, -1) for t in eval_df["positive"].tolist()]
valid    = [(q, g) for q, g in zip(queries, gt_idxs) if g >= 0]
queries_v, gt_v = zip(*valid)
print(f"Eval: {len(queries_v)} запросов | {len(corpus)} документов в корпусе")

In [ ]:
# ── 1. Baseline bi-encoder ───────────────────────────────────────────────────
print("Считаем эмбеддинги baseline...")
model_base = SentenceTransformer("deepvk/USER-bge-m3", device=DEVICE)
q_base = model_base.encode(list(queries_v), batch_size=64, normalize_embeddings=True,
                            show_progress_bar=True, convert_to_numpy=True)
c_base = model_base.encode(corpus, batch_size=64, normalize_embeddings=True,
                            show_progress_bar=True, convert_to_numpy=True)
sims_base = q_base @ c_base.T
r_base = recall_at_k(sims_base, gt_v)
print("Baseline:", r_base)

In [ ]:
# ── 2. Дообученный bi-encoder (TSDAE→MNRL) ──────────────────────────────────
bienc_path = str(BIENCODER_OUT / "stage2_tsdae_mnrl" / "best_mnrl")
import os.path
if os.path.exists(bienc_path):
    print(f"Загружаем дообученный bi-encoder: {bienc_path}")
    model_ft = SentenceTransformer(bienc_path, device=DEVICE)
    q_ft = model_ft.encode(list(queries_v), batch_size=64, normalize_embeddings=True,
                            show_progress_bar=True, convert_to_numpy=True)
    c_ft = model_ft.encode(corpus, batch_size=64, normalize_embeddings=True,
                            show_progress_bar=True, convert_to_numpy=True)
    sims_ft = q_ft @ c_ft.T
    r_ft = recall_at_k(sims_ft, gt_v)
    print("Bi-encoder (дообученный):", r_ft)
else:
    print(f"Путь не найден: {bienc_path}")
    print("Запустите сначала finetune_banking_embeddings.ipynb (TSDAE → MNRL)")
    sims_ft  = sims_base
    r_ft     = r_base

In [ ]:
# ── 3. Bi-encoder + Reranker ─────────────────────────────────────────────────
print(f"\nRerank: bi-encoder top-{RETRIEVE_K} → reranker → top-{RERANK_K}")
reranker_path = str(OUT / "best_reranker")
reranker = CrossEncoder(reranker_path, max_length=512, device=DEVICE)

def pipeline_recall(sims_matrix, queries, corpus, reranker, retrieve_k, rerank_k, gt_idxs, ks):
    """bi-encoder top-retrieve_k → cross-encoder → top-rerank_k."""
    top_preds = []
    for i, q in enumerate(queries):
        # Шаг 1: bi-encoder → top-retrieve_k кандидатов
        cand_idxs = np.argpartition(sims_matrix[i], -retrieve_k)[-retrieve_k:]
        cand_texts = [corpus[j] for j in cand_idxs]

        # Шаг 2: cross-encoder reranking
        pairs = [[q, doc] for doc in cand_texts]
        ce_scores = reranker.predict(pairs, show_progress_bar=False)

        # Шаг 3: top-rerank_k из кандидатов
        top_local = np.argpartition(ce_scores, -rerank_k)[-rerank_k:]
        top_global = set(cand_idxs[top_local])
        top_preds.append(top_global)

    results = {}
    for k in ks:
        hits = sum(1 for i, gt in enumerate(gt_idxs) if gt in top_preds[i])
        results[f"Recall@{k}"] = round(hits / len(gt_idxs), 4)
    return results

r_pipe = pipeline_recall(
    sims_ft, list(queries_v), corpus, reranker,
    retrieve_k=RETRIEVE_K, rerank_k=RERANK_K,
    gt_idxs=gt_v, ks=[1, 3, 5, 10],
)
print("Bi-encoder + Reranker:", r_pipe)

In [ ]:
# Сводная таблица и визуализация
metrics = ["Recall@1", "Recall@3", "Recall@5", "Recall@10"]

summary = pd.DataFrame({
    "Baseline bi-enc":      [r_base.get(m, 0) for m in metrics],
    "Fine-tuned bi-enc":    [r_ft.get(m, 0)   for m in metrics],
    f"Bi-enc+Reranker":     [r_pipe.get(m, 0)  for m in metrics],
}, index=metrics)

print("\n" + "="*60)
print(summary.to_string())

# Дельта
delta = summary[f"Bi-enc+Reranker"] - summary["Baseline bi-enc"]
print("\nДельта (Reranker vs Baseline):")
print(delta.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
summary.T.plot(kind="bar", ax=ax, width=0.65)
ax.set_title(f"Recall@k: baseline vs fine-tuned vs +reranker (retrieve={RETRIEVE_K}→rerank={RERANK_K})")
ax.set_xlabel("Модель / пайплайн")
ax.set_ylabel("Recall@k")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "reranker_comparison.png", dpi=120)
plt.show()

## 5. Анализ: что реранкер исправил

Смотрим на запросы, где bi-encoder промахнулся, а реранкер нашёл правильный ответ.


In [ ]:
fixed, broken = [], []
for i, (q, gt) in enumerate(zip(queries_v, gt_v)):
    top_bienc = set(np.argpartition(sims_ft[i], -RERANK_K)[-RERANK_K:])
    
    # Реранкер для этого запроса
    cand_idxs = np.argpartition(sims_ft[i], -RETRIEVE_K)[-RETRIEVE_K:]
    cand_texts = [corpus[j] for j in cand_idxs]
    ce_sc = reranker.predict([[q, d] for d in cand_texts], show_progress_bar=False)
    top_ce = set(cand_idxs[np.argpartition(ce_sc, -RERANK_K)[-RERANK_K:]])

    if gt not in top_bienc and gt in top_ce:
        fixed.append({"query": q, "correct_passage": corpus[gt][:150]})
    elif gt in top_bienc and gt not in top_ce:
        broken.append({"query": q, "correct_passage": corpus[gt][:150]})

print(f"Реранкер ИСПРАВИЛ {len(fixed)} запросов")
print(f"Реранкер СЛОМАЛ  {len(broken)} запросов")
print(f"Чистый прирост   : {len(fixed) - len(broken)}")

if fixed:
    print("\n── Примеры исправленных запросов ──")
    for ex in fixed[:2]:
        print(f"  Q: {ex['query'][:100]}")
        print(f"  A: {ex['correct_passage'][:100]}")
        print()